# Microsoft Agent Framework — Azure OpenAI (Responses API)

Selles näites kasutate **Microsoft Agent Frameworki (MAF)** lihtsa agendi loomiseks, mida toetab **Azure OpenAI** kasutades **Responses API**.

> **Migratsioonimärkus:** See näidis kasutas varem Semantic Kernelit koos GitHubi mudelitega. See on migratsioon tehtud Microsoft Agent Frameworki ning GitHubi mudelid (vanu, pensionile minek juulis 2026) on asendatud Azure OpenAIga, mis toetab Responses API-d. MAF-i `OpenAIChatClient` sihib Azure OpenAI stabiilset `/openai/v1/` lõpp-punkti ja kasutab vaikimisi Responses API-d.

Selle näite eesmärk on näidata samme, mida hakatakse hiljem kasutama teistes näidiskoodides erinevate agentsete mustrite rakendamisel.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Impordi vajalikud Python'i paketid


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Tööriista määratlemine

Microsoft Agent Frameworkis on **tööriist** tavaline Pythoni funktsioon, mida kaunistab `@tool` ja mida agent saab kutsuda. Allpool määratleme tööriista, mis tagastab juhusliku puhkuse sihtkoha ja väldib eelmise kordamist.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Agendi loomine

Siin loome agendi nimega `TravelAgent`.

Selles näites kasutame väga lihtsaid juhiseid. Muutke julgelt neid juhiseid, et jälgida, kuidas agendi käitumine muutub.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Agendi käivitamine

Nüüd saame agenti käivitada. Loome `AgentSession`i, et agent mäletaks vestlust mitme sammu jooksul, seejärel saadame kaks `user_inputs`i. Esimene küsib reisi kohta; teine ütleb, et kasutajale ei meeldinud ettepanek ja palub teise — agent kasutab vastamiseks sessiooni ajalugu ja `get_random_destination` tööriista.

Sa võid neid sõnumeid muuta, et näha, kuidas agent erinevalt reageerib. Vastused on **voogedastatavad** tokeni kaupa.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
